In [3]:
import duckdb, pandas as pd, os

print("Files in data:", os.listdir("data"))

con = duckdb.connect("olist.duckdb")

tables = {
    "orders": "olist_orders_dataset",
    "customers": "olist_customers_dataset",
    "order_items": "olist_order_items_dataset",
    "payments": "olist_order_payments_dataset",
    "reviews": "olist_order_reviews_dataset",
    "products": "olist_products_dataset",
    "sellers": "olist_sellers_dataset",
    "category_translation": "product_category_name_translation",
}

for name, file in tables.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE {name} AS
        SELECT * FROM read_csv_auto('data/{file}.csv')
    """)
    n = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(name, n)

Files in data: ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']
orders 99441
customers 99441
order_items 112650
payments 103886
reviews 99224
products 32951
sellers 3095
category_translation 71


In [4]:
con.execute("""
    SELECT MIN(order_purchase_timestamp) AS first_order,
           MAX(order_purchase_timestamp) AS last_order,
           COUNT(*) AS orders
    FROM orders
""").df()

,first_order,last_order,orders
0,2016-09-04 21:15:19,2018-10-17 17:30:18,99441


In [6]:
con.execute("""
    SELECT COUNT(DISTINCT customer_id)        AS distinct_customer_ids,
           COUNT(DISTINCT customer_unique_id) AS distinct_real_customers,
           COUNT(*)                           AS rows_in_customers
    FROM customers
""").df()

,distinct_customer_ids,distinct_real_customers,rows_in_customers
0,99441,96096,99441


In [7]:
print("Orders per status")
print(con.execute("""
    SELECT order_status, COUNT(*) AS orders
    FROM orders
    GROUP BY order_status
    ORDER BY orders DESC
""").df().to_string(index=False))

print("\nDelivered orders with no delivery date")
print(con.execute("""
    SELECT COUNT(*) AS missing_delivery_date
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NULL
""").df().to_string(index=False))

print("\nOrders with more than one payment row")
print(con.execute("""
    SELECT COUNT(*) AS orders_with_multiple_payments
    FROM (
        SELECT order_id
        FROM payments
        GROUP BY order_id
        HAVING COUNT(*) > 1
    )
""").df().to_string(index=False))

Orders per status
order_status  orders
   delivered   96478
     shipped    1107
    canceled     625
 unavailable     609
    invoiced     314
  processing     301
     created       5
    approved       2

Delivered orders with no delivery date
 missing_delivery_date
                     8

Orders with more than one payment row
 orders_with_multiple_payments
                          2961


In [8]:
con.execute("""
CREATE OR REPLACE TABLE delivered_orders AS
SELECT
    o.order_id,
    c.customer_unique_id,
    c.customer_state,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    date_diff('day', o.order_estimated_delivery_date,
                     o.order_delivered_customer_date) AS days_vs_estimate,
    (o.order_delivered_customer_date > o.order_estimated_delivery_date) AS is_late
FROM orders o
JOIN customers c USING (customer_id)
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TABLE reviews_clean AS
SELECT order_id, review_score
FROM reviews
QUALIFY row_number() OVER (
    PARTITION BY order_id
    ORDER BY review_creation_date DESC, review_answer_timestamp DESC
) = 1
""")

con.execute("""
SELECT COUNT(*) AS delivered_orders,
       SUM(is_late::INT) AS late_orders,
       ROUND(100.0 * AVG(is_late::INT), 2) AS late_pct
FROM delivered_orders
""").df()

,delivered_orders,late_orders,late_pct
0,96470,7826.0,8.11


In [ ]:
from scipy.stats import mannwhitneyu

df = con.execute("""
    SELECT d.order_id, d.is_late, d.days_vs_estimate,
           d.customer_state, r.review_score
    FROM delivered_orders d
    JOIN reviews_clean r USING (order_id)
""").df()

print(df.groupby("is_late")["review_score"].agg(["count", "mean", "median"]))

late = df[df.is_late]["review_score"]
on_time = df[~df.is_late]["review_score"]

u, p = mannwhitneyu(late, on_time, alternative="two-sided")
n1, n2 = len(late), len(on_time)
print("p-value:", p)
print("rank-biserial effect size:", round(2 * u / (n1 * n2) - 1, 3))

df["low_score"] = df.review_score <= 2
print(df.groupby("is_late")["low_score"].mean().round(3))